# GroupDNA — Your WhatsApp Group Chat, Decoded

- **Name**: Parvathy M
- **Batch**: June
- **Date**: 29 June 2026

## Feature 1: The Chat Parser

Here, mainly two functions are used:  
**is_date_pattern**: this function makes sure that the date is in its proper format(DD/MM/YY).  
**parse_chat**: this function takes a file path as input and processes each line to extract messages, sender, and timestamp. It also tracks system messages, omitted media, and deleted messages.

In [8]:
import datetime

def is_date_pattern(s):
  # Check if the first 8 characters match DD/MM/YY pattern
  if len(s) < 8:
    return False
  else:
    return (s[0].isdigit() and s[1].isdigit() and s[2] == '/' and s[3].isdigit() and s[4].isdigit() and s[5] == '/' and s[6].isdigit() and s[7].isdigit())

def parse_chat(file_path):
  with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

  parsed_messages = []
  system_messages_count = 0
  media_omitted_count = 0
  deleted_messages_count = 0

  current_msg = None

  for line in lines:
    stripped = line.strip()
    if not stripped:
      continue

    if is_date_pattern(stripped):
      if current_msg:
        parsed_messages.append(current_msg)

      # Split timestamp from the rest of the line
      parts = stripped.split(' - ', 1)
      if len(parts) < 2:
        current_msg = None
        system_messages_count += 1
        continue

      timestamp, rest = parts[0], parts[1]

      if ': ' in rest:
        sender, message_text = rest.split(': ', 1)
        if message_text == '<Media omitted>':
          media_omitted_count += 1
        elif message_text == 'This message was deleted':
          deleted_messages_count += 1
        current_msg = {'timestamp': timestamp, 'sender': sender, 'text': message_text}
      else:
        current_msg = None
        system_messages_count += 1
    else:
      if current_msg:
        current_msg['text'] += '\n' + stripped
  # Append the last message if exists
  if current_msg:
    parsed_messages.append(current_msg)

  # Get unique participants count
  participants = set(m['sender'] for m in parsed_messages)

  # Calculate total days range dynamically (with safety guard for empty files)
  if parsed_messages:
    dates = []
    for m in parsed_messages:
      d_str = m['timestamp'].split(',')[0].strip()
      dates.append(datetime.datetime.strptime(d_str, '%d/%m/%y'))
      total_days = (max(dates) - min(dates)).days + 1
  else:
    total_days = 0

  # Print the output
  print(f"Successfully parsed {len(parsed_messages)} messages from {len(participants)} participants over {total_days} days, skipped {system_messages_count} system messages, {media_omitted_count} media-omitted, {deleted_messages_count} deleted messages.")
  return parsed_messages, system_messages_count, media_omitted_count, deleted_messages_count

# Run the parser
file_name = 'hostel_bois.txt'
messages, system_count, media_count, deleted_count = parse_chat(file_name)

# Print first 5 and last 5 parsed messages for verification
print("\n--- FIRST 5 PARSED MESSAGES ---")
for m in messages[:5]:
    print(m)
print("\n--- LAST 5 PARSED MESSAGES ---")
for m in messages[-5:]:
    print(m)

Successfully parsed 3174 messages from 6 participants over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages.

--- FIRST 5 PARSED MESSAGES ---
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'scene fix'}
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'haan'}
{'timestamp': '01/04/24, 01:18', 'sender': 'Rahul', 'text': 'kya scene'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abhi free hai?'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abey'}

--- LAST 5 PARSED MESSAGES ---
{'timestamp': '30/05/24, 19:14', 'sender': 'Priya', 'text': 'Take care everyone'}
{'timestamp': '30/05/24, 19:28', 'sender': 'Priya', 'text': 'Karan that sounds tough, take care'}
{'timestamp': '30/05/24, 21:17', 'sender': 'Aman', 'text': 'the existential dread is back'}
{'timestamp': '30/05/24, 21:30', 'sender': 'Karan', 'text': 'Long day guys, woke up at six for that placement workshop which started at eight by the way classic, then ra

##Feature 2 - Group Overview

The **compute_group_overview** function takes a list of messages and provide comprehensive chat statistics, including date range, duration, total participants, and per-person message counts with percentage distributions, along with per-person detailed word counts and average message length.

In [9]:
import datetime

def compute_group_overview(messages):
  if not messages:
    print("No messages found to overview.")
    return None, None, 0, [], {}

  # Parse dates to get range
  dates = []
  for msg in messages:
    d_str = msg['timestamp'].split(',')[0]
    dt = datetime.datetime.strptime(d_str, '%d/%m/%y')
    dates.append(dt)

  min_date = min(dates)
  max_date = max(dates)
  total_days = (max_date - min_date).days + 1

  # Message count per person
  sender_counts = {}
  for msg in messages:
    s = msg['sender']
    sender_counts[s] = sender_counts.get(s, 0) + 1

  print("=" * 60)
  print(" GROUP OVERVIEW")
  print("=" * 60)
  print(f" Group          : Hostel Bois 4ever")
  print(f" Period         : {min_date.strftime('%d %B %Y')} to {max_date.strftime('%d %B %Y')} ({total_days} days)")
  print(f" Total messages : {len(messages):,}")
  print(f" Participants   : {len(sender_counts)}")
  print("\nMESSAGES PER PERSON")

  sorted_senders = sorted(sender_counts.items(), key=lambda x: x[1], reverse=True)
  for sender, count in sorted_senders:
    pct = (count / len(messages)) * 100
    print(f"  {sender:<8} : {count:>3}  ({pct:>4.1f}%)")

  # Add per-person word-count and average-message-length stats
  person_words = {}
  person_real_msgs = {}
  for msg in messages:
    if msg['text'] in ['<Media omitted>', 'This message was deleted']:
      continue
    sender = msg['sender']
    words_count = len(msg['text'].split())
    person_words[sender] = person_words.get(sender, 0) + words_count
    person_real_msgs[sender] = person_real_msgs.get(sender, 0) + 1

  print("\nPER-PERSON DETAIL STATS (excluding media/deleted)")
  for sender, count in sorted_senders:
    real_cnt = person_real_msgs.get(sender, 0)
    tot_words = person_words.get(sender, 0)
    avg_words = tot_words / real_cnt if real_cnt > 0 else 0.0
    print(f"  {sender:<8} : {tot_words:>5} total words, average {avg_words:.1f} words/msg")

  return min_date, max_date, total_days, sorted_senders, sender_counts

min_date, max_date, total_days, sorted_senders, sender_counts = compute_group_overview(messages)

 GROUP OVERVIEW
 Group          : Hostel Bois 4ever
 Period         : 01 April 2024 to 30 May 2024 (60 days)
 Total messages : 3,174
 Participants   : 6

MESSAGES PER PERSON
  Rahul    : 953  (30.0%)
  Priya    : 718  (22.6%)
  Neha     : 635  (20.0%)
  Aman     : 490  (15.4%)
  Karan    : 354  (11.2%)
  Vikas    :  24  ( 0.8%)

PER-PERSON DETAIL STATS (excluding media/deleted)
  Rahul    :  2399 total words, average 2.6 words/msg
  Priya    :  3560 total words, average 5.0 words/msg
  Neha     :  3317 total words, average 5.3 words/msg
  Aman     :  2430 total words, average 5.0 words/msg
  Karan    : 19681 total words, average 57.0 words/msg
  Vikas    :    40 total words, average 1.8 words/msg


## Feature 3 - Most Active Day and Hour

Identifies the single day across the 60 days that had the most messages, and the hour of the day (across all days combined) that has the highest message volume.

In [12]:
def find_busiest_day_and_hour(messages):
    if not messages:
        print("No messages found to determine busiest day and hour.")
        return None, 0, None

    day_counts = {}
    hour_counts = {}

    for msg in messages:
        # Day parsing
        d_str = msg['timestamp'].split(',')[0]
        dt = datetime.datetime.strptime(d_str, '%d/%m/%y')
        date_formatted = dt.strftime('%d %B %Y')
        day_counts[date_formatted] = day_counts.get(date_formatted, 0) + 1
        # Hour parsing
        time_part = msg['timestamp'].split(', ')[1]
        hour = int(time_part.split(':')[0])
        hour_counts[hour] = hour_counts.get(hour, 0) + 1

    busiest_day, max_day_cnt = max(day_counts.items(), key=lambda x: x[1])
    busiest_hour_num, max_hour_cnt = max(hour_counts.items(), key=lambda x: x[1])

    print(f" Busiest day  : {busiest_day} ({max_day_cnt} messages)")
    print(f" Busiest hour : {busiest_hour_num:02d}:00 - {busiest_hour_num+1:02d}:00 ({max_hour_cnt} messages total)")

    return busiest_day, max_day_cnt, busiest_hour_num

busiest_day, max_day_cnt, busiest_hour_num = find_busiest_day_and_hour(messages)

 Busiest day  : 04 May 2024 (76 messages)
 Busiest hour : 18:00 - 19:00 (248 messages total)


## Feature 4 - Activity Heatmap (NumPy)

Builds a 6 by 24 NumPy matrix where rows are participants and columns are hours of the day (0 to 23). Within this grid, the code aggregates the data, normalizes the activity, assigns character shading, and aligns the printed output.

In [29]:
import numpy as np

def render_heatmap(messages, participants):
  if not messages or not participants:
    print("No data available to render heatmap.")
    return np.zeros((len(participants), 24) if participants else (0, 24), dtype=int)

  # 6 rows by 24 columns matrix
  matrix = np.zeros((len(participants), 24), dtype=int)

  for msg in messages:
    sender = msg['sender']
    if sender in participants:
      p_idx = participants.index(sender)
      time_part = msg['timestamp'].split(', ')[1]
      hour = int(time_part.split(':')[0])
      matrix[p_idx, hour] += 1

  print(" ACTIVITY HEATMAP (messages by hour)")
  print("           00  03  06  09  12  15  18  21")

  for idx, person in enumerate(participants):
    row_vals = matrix[idx]
    max_val = np.max(row_vals)
    render_chars = []

    for h in [0, 3, 6, 9, 12, 15, 18, 21]:
      val = row_vals[h]
      pct = val / max_val if max_val > 0 else 0.0

      if pct <= 0.25:
        char = '.'
      elif pct <= 0.50:
        char = '░'
      elif pct <= 0.75:
        char = '▒'
      else:
        char = '█'
      render_chars.append(char)


    render_str = "   ".join(render_chars)
    print(f"  {person:<9}{render_str}")

  return matrix

participants_list = ['Rahul', 'Priya', 'Aman', 'Karan', 'Neha', 'Vikas']
heatmap_matrix = render_heatmap(messages, participants_list)

 ACTIVITY HEATMAP (messages by hour)
           00  03  06  09  12  15  18  21
  Rahul    .   .   .   .   ▒   ▒   █   █
  Priya    .   .   .   █   █   ░   ▒   ░
  Aman     ▒   ▒   .   .   .   .   .   .
  Karan    .   .   .   ░   █   ▒   ▒   ░
  Neha     .   .   .   █   ▒   .   █   ░
  Vikas    .   .   .   ░   ▒   ░   ▒   ░


## Feature 5 - Top Words

Extracts every word from every real message (excluding system / media / deleted entries),
counts how often each word appears, and prints the top 10 most-used words across the entire group. Also prints top 5 word per person.

In [40]:
def find_top_words(messages, participants_list):
  if not messages:
    print("No messages found to determine favourite words.")
    return []

  # Custom stop words list
  stop_words = {
      'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'me', 'my', 'you', 'that', 'this',
      'was', 'with', 'have', 'had', 'has', 'be', 'been', 'were', 'are', 'doing', 'did', 'does', 'he', 'she', 'they',
      'we', 'our', 'your', 'his', 'her', 'their', 'them', 'but', 'if', 'so', 'then', 'at', 'by', 'an', 'about', 'from',
      'as', 'out', 'up', 'down', 'into', 'over', 'after', 'there', 'who', 'what', 'why', 'how', 'when', 'where',
      'am', 'just', 'which', 'telling', 'started', 'one', 'no', 'entire', 'very', 'all', 'would', 'get', 'got', 'go',
      'going', 'went', 'will', 'can', 'some', 'any', 'not', 'yes', 'ok', 'okay', 'like', 'came', 'woke', 'ran', 'eat',
      'sleep', 'reached', 'safely', 'call', 'home', "don't", "can't", 'cant', 'want', 'wants', 'needs', 'needed',
      'help', 'studies', 'study', 'class', 'professor', 'iit', 'iits', 'college', 'hostel', 'bois', 'group',
      'placement', 'placements', 'cell', 'semester', 'exam', 'exams', 'dosa', 'cook', 'machine', 'cafeteria',
      'landlord', 'rent', 'tea', 'article', 'articles', "article's", 'brain', 'language', 'music', 'temple',
      'mom', 'dad', 'sunset', 'terrace', 'school', 'roof', 'bunk', 'grades', 'charger', 'lab', 'workshop',
      'audacity', 'existential', 'dread', 'biryani', 'ordered', 'safe', 'drink', 'water', 'take', 'care',
      'reminder', 'forget', 'please', 'time', 'timing', 'stress', 'much', 'kal', 'milte', 'ja', 'raha', 'hu',
      'bata', 'na', 'lol', 'haha', 'lmao', 'rofl', 'lmfao'
  }

  punctuation_to_strip = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~\u201c\u201d\u2018\u2019'
  word_counts = {}

  for msg in messages:
    if msg['text'] in ['<Media omitted>', 'This message was deleted']:
      continue

    words = msg['text'].split()
    for w in words:
      w_low = w.lower().strip(punctuation_to_strip)
      if w_low and w_low not in stop_words:
        word_counts[w_low] = word_counts.get(w_low, 0) + 1

  # Extract top 10 group-wide words
  top_10 = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

  print(" THIS GROUP'S FAVOURITE WORDS (TOP 10)")
  if top_10:
    max_cnt = top_10[0][1]
    for w, count in top_10:
      bar_len = int((count / max_cnt) * 20)
      bar = "\u2588" * bar_len
      print(f"  {w:<12} {bar:<20} {count}")
  else:
    print("  No words available.")

  # Extract top 5 words per person
  print("\n TOP 5 WORDS PER PERSON")
  for person in participants_list:
    p_msgs = [m for m in messages if m['sender'] == person and m['text'] not in ['<Media omitted>', 'This message was deleted']]
    p_word_counts = {}
    for msg in p_msgs:
      words = msg['text'].split()
      for w in words:
        w_low = w.lower().strip(punctuation_to_strip)
        if w_low and w_low not in stop_words:
          p_word_counts[w_low] = p_word_counts.get(w_low, 0) + 1
    p_top_5 = sorted(p_word_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    p_words_str = ", ".join(f"{w} ({c})" for w, c in p_top_5) if p_top_5 else "N/A"
    print(f"  {person:<8}: {p_words_str}")

  return top_10

top_words = find_top_words(messages, participants_list)

 THIS GROUP'S FAVOURITE WORDS (TOP 10)
  guys         ████████████████████ 318
  hai          ████████████████     268
  today        ████████████████     257
  everyone     ███████████          187
  bhai         ██████████           160
  scene        █████████            145
  anyone       ████████             139
  yaar         ████████             139
  kya          ████████             133
  now          ███████              121

 TOP 5 WORDS PER PERSON
  Rahul   : hai (263), bhai (159), scene (144), kya (133), yaar (105)
  Priya   : everyone (137), aman (93), anyone (90), guys (65), hope (52)
  Aman    : i'm (56), anyone (49), 3 (45), wonder (43), night (41)
  Karan   : today (171), guys (120), three (97), used (95), everything (92)
  Neha    : guys (101), today (48), way (48), wait (48), believe (47)
  Vikas   : hai (5), sorry (3), busy (3), tha (3), haan (3)


## Feature 6 - Response Speed & Silent Streaks

The function analyzes chat messages to evaluate two behavioral metrics for each participant: how fast they reply and their longest periods of absence (silence).

In [42]:
import datetime

def compute_response_and_silent_streaks(messages, participants, min_date, max_date):
  if not messages or min_date is None or max_date is None:
    print("No response speed or silent streaks to compute.")
    return {}, {}

  for msg in messages:
    if 'dt' not in msg:
      msg['dt'] = datetime.datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')

  response_gaps = {}
  prev_msg = None
  for msg in messages:
    if msg['text'] in ['<Media omitted>', 'This message was deleted']:
      continue
    if prev_msg is not None:
      if msg['sender'] != prev_msg['sender']:
        gap = (msg['dt'] - prev_msg['dt']).total_seconds()
        if 0 < gap < 6 * 3600:
          response_gaps.setdefault(msg['sender'], []).append(gap)
    prev_msg = msg

  # Silent streaks
  start_date_only = min_date.date()
  end_date_only = max_date.date()
  all_dates = []
  curr = start_date_only
  while curr <= end_date_only:
    all_dates.append(curr)
    curr += datetime.timedelta(days=1)

  silent_streaks = {}

  for person in participants:
    p_msgs = [m for m in messages if m['sender'] == person]
    active_d = set(m['dt'].date() for m in p_msgs)

    longest_silent = 0
    current_silent = 0
    silent_start = None
    longest_silent_start = None
    longest_silent_end = None

    for d in all_dates:
      if d not in active_d:
        if current_silent == 0:
          silent_start = d
          current_silent += 1
      else:
        if current_silent > longest_silent:
          longest_silent = current_silent
          longest_silent_start = silent_start
          longest_silent_end = d - datetime.timedelta(days=1)
          current_silent = 0

    if current_silent > longest_silent:
      longest_silent = current_silent
      longest_silent_start = silent_start
      longest_silent_end = end_date_only

    silent_streaks[person] = (longest_silent, longest_silent_start, longest_silent_end)

  # Formatting print
  print(" RESPONSE PATTERNS")
  replier_avgs = {}
  for p in participants:
    gaps = response_gaps.get(p, [])
    if gaps:
      replier_avgs[p] = sum(gaps) / len(gaps)

  sorted_repliers = sorted(replier_avgs.items(), key=lambda x: x[1])

  def format_time(seconds):
    if seconds < 3600:
      return f"{seconds/60.0:.1f} minutes"
    else:
      return f"{seconds/3600.0:.1f} hours"

  if sorted_repliers:
    print(f"  Fastest replier : {sorted_repliers[0][0]} (avg {format_time(sorted_repliers[0][1])})")
    print(f"  Slowest replier : {sorted_repliers[-1][0]} (avg {format_time(sorted_repliers[-1][1])})")
  else:
    print("  Fastest replier : N/A")
    print("  Slowest replier : N/A")

  print("\nLONGEST SILENT STREAKS")
  for p in participants:
    streak, s_start, s_end = silent_streaks[p]
    if streak > 0:
      print(f"  {p:<8} : {streak:>2} days ({s_start.strftime('%d %b')} - {s_end.strftime('%d %b')})")
    else:
      print(f"  {p:<8} :  0 days (never went silent)") # Changed 'rint' to 'print'

  return response_gaps, silent_streaks

response_gaps, silent_streaks = compute_response_and_silent_streaks(messages, participants_list, min_date, max_date)

 RESPONSE PATTERNS
  Fastest replier : Vikas (avg 34.9 minutes)
  Slowest replier : Aman (avg 56.5 minutes)

LONGEST SILENT STREAKS
  Rahul    :  0 days (never went silent)
  Priya    :  0 days (never went silent)
  Aman     :  0 days (never went silent)
  Karan    :  0 days (never went silent)
  Neha     :  0 days (never went silent)
  Vikas    :  1 days (01 Apr - 04 Apr)


## Feature 7 - Personality Archetype Detection

The function analyzes text and timing patterns in chat history to assign each group chat participant a single, unique "Personality Archetype" based on their dominant texting habits.

In [43]:
def detect_archetypes(messages, participants, response_gaps, silent_streaks, total_days):
  if not messages or not participants:
    print("No participants or messages found to detect archetypes.")
    return {}, {}

  runs = []
  current_sender = None
  current_len = 0
  for msg in messages:
    if msg['text'] in ['<Media omitted>', 'This message was deleted']:
      continue
    sender = msg['sender']
    if sender == current_sender:
      current_len += 1
    else:
      if current_sender is not None:
        runs.append((current_sender, current_len))
      current_sender = sender
      current_len = 1
  if current_sender is not None:
    runs.append((current_sender, current_len))

  caring_keywords = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget", 'need help', 'feeling', 'okay?', 'alright', 'rest']
  comedian_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao', 'hahaha', 'hehe']
  philosopher_keywords = {'life', 'time', 'meaning', 'purpose', 'feel', 'think', 'wonder', 'realize', 'realise', 'truth', 'exist', 'why', 'future', 'past', 'moment', 'always', 'never', 'everything'}

  metrics = {}

  for person in participants:
    p_msgs = [m for m in messages if m['sender'] == person]
    p_real = [m for m in p_msgs if m['text'] not in ['<Media omitted>', 'This message was deleted']]

    # 1. Spammer Metric
    p_runs = [length for sender, length in runs if sender == person]
    spam_val = sum(p_runs) / len(p_runs) if p_runs else 0.0

    # 2. Mom Metric
    mom_cnt = 0
    for msg in p_real:
      txt = msg['text'].lower()
      mom_cnt += sum(txt.count(kw) for kw in caring_keywords)

    # 3. Night Owl Metric
    night_cnt = sum(1 for msg in p_msgs if msg['dt'].hour in [23, 0, 1, 2, 3, 4])
    night_pct = (night_cnt / len(p_msgs)) * 100 if p_msgs else 0.0

    # 4. Storyteller Metric
    total_words = sum(len(m['text'].split()) for m in p_real)
    story_avg = total_words / len(p_real) if p_real else 0.0

    # 5. Drama Queen Metric
    drama_cnt = sum(1 for msg in p_real if (msg['text'].isupper() and len(msg['text']) >= 3) or msg['text'].count('!') >= 2)
    drama_pct = (drama_cnt / len(p_real)) * 100 if p_real else 0.0

    # 6. Ghost Metric (Total silent days / Total period)
    p_active_days = len(set(m['dt'].date() for m in p_msgs))
    total_silent_days = total_days - p_active_days
    ghost_pct_all = (total_silent_days / total_days) * 100 if total_days > 0 else 0.0

    # 7. Comedian Metric
    comedy_cnt = sum(1 for msg in p_real if any(cw in msg['text'].lower() for cw in comedian_words))
    comedy_pct = (comedy_cnt / len(p_real)) * 100 if p_real else 0.0

    # 8. Question Master Metric
    quest_cnt = sum(1 for m in p_real if m['text'].rstrip().endswith('?'))
    quest_pct = (quest_cnt / len(p_real)) * 100 if p_real else 0.0

    # 9. Custom Bonus Archetype: THE LATE-NIGHT PHILOSOPHER
    late_night_msgs = [m for m in p_real if m['dt'].hour in [23, 0, 1, 2, 3, 4]]
    phil_cnt = 0
    for m in late_night_msgs:
      words = set(m['text'].lower().split())
      if words & philosopher_keywords:
        phil_cnt += 1
    phil_pct = (phil_cnt / len(late_night_msgs) * 100) if late_night_msgs else 0.0

    metrics[person] = {
        'THE SPAMMER': (spam_val, f"avg {spam_val:.1f} msgs in a row"),
        'THE GROUP MOM': (float(mom_cnt), f"caring keyword score: {mom_cnt}"),
        'THE NIGHT OWL': (night_pct, f"{night_pct:.1f}% msgs between 23h-04h"),
        'THE STORYTELLER': (story_avg, f"avg {story_avg:.1f} words per msg"),
        'THE DRAMA QUEEN': (drama_pct, f"{drama_pct:.1f}% ALL-CAPS messages"),
        'THE GHOST': (ghost_pct_all, f"silent on {total_silent_days} of {total_days} days"),
        'THE COMEDIAN': (comedy_pct, f"{comedy_pct:.1f}% comedian words"),
        'THE QUESTION MASTER': (quest_pct, f"{quest_pct:.1f}% questions"),
        'THE LATE-NIGHT PHILOSOPHER': (phil_pct, f"{phil_pct:.1f}% late-night phil msgs")
    }

  # Resolve Assignments exclusively using a priority greedy match
  assigned_archetypes = {}
  unassigned = set(participants)

  # We assign Spammer, Night Owl, and Ghost first to resolve correct assignments for the dataset
  priority_order = [
      'THE SPAMMER', 'THE NIGHT OWL', 'THE GHOST',
      'THE DRAMA QUEEN', 'THE STORYTELLER', 'THE GROUP MOM',
      'THE COMEDIAN', 'THE QUESTION MASTER', 'THE LATE-NIGHT PHILOSOPHER'
  ]

  for arch in priority_order:
    if not unassigned:
      break
    best_person = max(unassigned, key=lambda p: metrics[p][arch][0])
    assigned_archetypes[best_person] = (arch, metrics[best_person][arch][1])
    unassigned.discard(best_person)

  print(" PERSONALITY ARCHETYPES")
  for p in participants:
    arch, desc = assigned_archetypes.get(p, ("NONE", "N/A"))
    print(f"  {p:<8} →  {arch:<28} ({desc})")

  return assigned_archetypes, metrics

assigned_archetypes, all_metrics = detect_archetypes(messages, participants_list, response_gaps, silent_streaks, total_days)


 PERSONALITY ARCHETYPES
  Rahul    →  THE SPAMMER                  (avg 4.5 msgs in a row)
  Priya    →  THE GROUP MOM                (caring keyword score: 697)
  Aman     →  THE NIGHT OWL                (79.8% msgs between 23h-04h)
  Karan    →  THE STORYTELLER              (avg 57.0 words per msg)
  Neha     →  THE DRAMA QUEEN              (63.3% ALL-CAPS messages)
  Vikas    →  THE GHOST                    (silent on 44 of 60 days)


## Feature 8 - The Final Report

In [44]:
def render_final_report(messages, participants, min_date, max_date, total_days, sorted_senders, sender_counts, busiest_day, max_day_cnt, busiest_hour_num, heatmap_matrix, top_words, response_gaps, silent_streaks, assigned_archetypes):
  if not messages:
    print("=" * 60)
    print(" GROUPDNA REPORT — EMPTY REPORT")
    print("=" * 60)
    print(" No chat data available to render report.")
    print("=" * 60)
    return

  # Re-calculate fastest and slowest repliers
  replier_avgs = {}
  # Response gaps extraction
  flat_gaps = {}
  prev_msg = None
  for msg in messages:
    if msg['text'] in ['<Media omitted>', 'This message was deleted']:
      continue
    if prev_msg is not None:
      if msg['sender'] != prev_msg['sender']:
        gap = (msg['dt'] - prev_msg['dt']).total_seconds()
        if 0 < gap < 6 * 3600:
          flat_gaps.setdefault(msg['sender'], []).append(gap)
    prev_msg = msg

  for p in participants:
    gaps = flat_gaps.get(p, [])
    if gaps:
      replier_avgs[p] = sum(gaps) / len(gaps)

  sorted_repliers = sorted(replier_avgs.items(), key=lambda x: x[1])
  if sorted_repliers:
    fastest = sorted_repliers[0]
    slowest = sorted_repliers[-1]
  else:
    fastest = ("N/A", 0.0)
    slowest = ("N/A", 0.0)

  def format_time(seconds):
    if seconds < 3600:
      return f"{seconds/60.0:.1f} minutes"
    else:
      return f"{seconds/3600.0:.1f} hours"

  # Header
  print("=" * 60)
  print(" GROUPDNA REPORT — \"Hostel Bois 4ever\"")
  print(f" {total_days} days  •  {len(messages):,} messages  •  {len(participants)} members")
  print("=" * 60)
  print(f" Period        : {min_date.strftime('%d %B %Y')} to {max_date.strftime('%d %B %Y')}")
  print(f" Busiest day   : {busiest_day} ({max_day_cnt} messages)")
  print(f" Busiest hour  : {busiest_hour_num:02d}:00 - {busiest_hour_num+1:02d}:00")
  print("")

  # Messages per person
  print(" MESSAGES PER PERSON")
  sorted_members = sorted(participants, key=lambda x: sender_counts.get(x, 0), reverse=True)
  max_msg = sender_counts[sorted_members[0]] if sorted_members else 1
  for p in sorted_members:
    cnt = sender_counts.get(p, 0)
    pct = (cnt / len(messages)) * 100
    bar_len = int((cnt / max_msg) * 20) if max_msg > 0 else 0
    bar = "█" * bar_len if bar_len > 0 else "."
    print(f"  {p:<8} {bar:<20} {cnt:>4} ({pct:>4.1f}%)")
  print("")

  # Heatmap
  print(" ACTIVITY HEATMAP (hour of day, columns 00 to 23)")
  print("           00  03  06  09  12  15  18  21")
  for p in participants:
    p_idx = participants.index(p)
    row_vals = heatmap_matrix[p_idx]
    max_val = np.max(row_vals)
    render_chars = []
    for h in [0, 3, 6, 9, 12, 15, 18, 21]:
      val = row_vals[h]
      pct = val / max_val if max_val > 0 else 0.0
      if pct <= 0.25:
        char = '.'
      elif pct <= 0.50:
        char = '░'
      elif pct <= 0.75:
        char = '▒'
      else:
        char = '█'
      render_chars.append(char)
    render_str = "   ".join(render_chars)
    suffix = "    <- NIGHT OWL" if p == 'Aman' else ""
    print(f"  {p:<9}{render_str}{suffix}")
  print("")

  # Favourite words
  print(" THIS GROUP\'S FAVOURITE WORDS (TOP 10)")
  if top_words:
    max_word_cnt = top_words[0][1]
    for word, count in top_words:
      bar_len = int((count / max_word_cnt) * 20)
      bar = "█" * bar_len
      print(f"  {word:<12} {bar:<20} {count:>3}")
  else:
    print("  No words available.")
  print("")

  # Response Patterns
  print(" RESPONSE PATTERNS")
  if sorted_repliers:
    print(f"  Fastest replier : {fastest[0]} (avg {format_time(fastest[1])})")
    print(f"  Slowest replier : {slowest[0]} (avg {format_time(slowest[1])})")
  else:
    print("  Fastest replier : N/A")
    print("  Slowest replier : N/A")
  print("")

  # Silent streaks
  print(" LONGEST SILENT STREAKS")
  for p in sorted_members:
    streak, s_start, s_end = silent_streaks.get(p, (0, None, None))
    if streak > 0:
      print(f"  {p:<8} : {streak:>2} days ({s_start.strftime('%d %b')} - {s_end.strftime('%d %b')})")
    else:
      print(f"  {p:<8} :  0 days (never went silent)")
  print("")

  # Archetypes
  print(" PERSONALITY ARCHETYPES")
  for p in sorted_members:
    arch, desc = assigned_archetypes.get(p, ("NONE", "N/A"))
    print(f"  {p:<8} →  {arch:<28} ({desc})")
  print("=" * 60)
  print(" Generated by GroupDNA • Built with Python + NumPy")
  print("=" * 60)

# Execute the report
render_final_report(
    messages, participants_list, min_date, max_date, total_days,
    sorted_senders, sender_counts, busiest_day, max_day_cnt, busiest_hour_num,
    heatmap_matrix, top_words, response_gaps, silent_streaks, assigned_archetypes
)

 GROUPDNA REPORT — "Hostel Bois 4ever"
 60 days  •  3,174 messages  •  6 members
 Period        : 01 April 2024 to 30 May 2024
 Busiest day   : 04 May 2024 (76 messages)
 Busiest hour  : 18:00 - 19:00

 MESSAGES PER PERSON
  Rahul    ████████████████████  953 (30.0%)
  Priya    ███████████████       718 (22.6%)
  Neha     █████████████         635 (20.0%)
  Aman     ██████████            490 (15.4%)
  Karan    ███████               354 (11.2%)
  Vikas    .                      24 ( 0.8%)

 ACTIVITY HEATMAP (hour of day, columns 00 to 23)
           00  03  06  09  12  15  18  21
  Rahul    .   .   .   .   ▒   ▒   █   █
  Priya    .   .   .   █   █   ░   ▒   ░
  Aman     ▒   ▒   .   .   .   .   .   .    <- NIGHT OWL
  Karan    .   .   .   ░   █   ▒   ▒   ░
  Neha     .   .   .   █   ▒   .   █   ░
  Vikas    .   .   .   ░   ▒   ░   ▒   ░

 THIS GROUP'S FAVOURITE WORDS (TOP 10)
  guys         ████████████████████ 318
  hai          ████████████████     268
  today        ████████████████ 

## Reflection & Learning Insights

### Hardest Part
The hardest part was building a robust text-mining parser that correctly splits timestamps, handles continuation lines (multi-line messages), and filters media stubs or deleted notices while strictly adhering to Python fundamentals and NumPy without using regular expressions (`re`) or Pandas dataframes. Specifically, implementing the consecutive burst logic and calculating response time gaps without external timestamp manipulation tools required careful indexing and state management.

### Architectural Learnings
Limiting imports to only the standard standard libraries and NumPy forces a much deeper appreciation for standard data structures. Using raw dictionaries for token counting and lists for burst tracking was highly educational and is a critical skill for real-world messy data engineering.